<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/10-ml-system-design/01-designing-an-inference-service.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Designing an Inference Service (concept)

**Goal:** Work an ML-system-design interview end to end. Turn a vague "design an LLM feature" prompt into a sized, defensible serving system: requirements, the back-of-envelope numbers (QPS / VRAM / latency / cost), the architecture, and the trade-offs you'll defend under questioning.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

> **This is a concept notebook, with no code to run.** System design is a *conversation*, not a program: the deliverable is a structured verbal answer and a whiteboard, so this notebook is the method and a fully worked example. The math it leans on is the sizing arithmetic you already ran in [09/02](../09-serving-inference/02-inference-performance.ipynb); here you *apply* it under interview pressure.

## Why an AI engineer gets a system-design round

Every other section built a *component*: a RAG pipeline, an agent loop, an eval harness. This round asks the next question: **can you deploy it as a service that survives real traffic, a budget, and an SLA?** It's the round that separates "I called an API in a notebook" from "I can own the serving layer of an AI product", the AI-distributed-systems job this track is aimed at.

The good news: you already have every ingredient. Serving frameworks (09/01), the throughput/latency levers and sizing math (09/02), reliability patterns (08/02), evals (02/04), cost hygiene (00/01). This notebook is the **connective tissue**: the order to raise them in, and how they compose into one defensible design.

> **⭐ Key takeaway —** interviewers don't score the *answer*; they score the *method*. A candidate who says "let me get the requirements and do the math first" and lands a *rough* number beats one who name-drops Kubernetes and never sizes anything. Structure is the skill.

## The method — a five-step loop

Use this skeleton for any "design an inference service" prompt. It's deliberately the same shape as the 09/02 sizing moves, expanded into a full design.

1. **Scope & requirements:** pin down the ask before designing. Functional (what does it do?) and non-functional (**latency target, QPS, cost ceiling, quality bar**). *Drive this like a discovery call (section 11): the vague prompt is intentional.*
2. **Back-of-envelope sizing:** QPS → tokens/sec → GPUs; VRAM per GPU; rough monthly cost. Numbers first, boxes second.
3. **Architecture:** the boxes and arrows: client → gateway → queue → model servers → cache, plus where evals and observability tap in.
4. **Trade-offs & bottlenecks:** name what breaks first and the lever you'd pull (this is where 09/02 pays off).
5. **Iterate to the follow-ups.** The interviewer will push ("10× the traffic", "cut cost in half", "p99 spiked"); each is a lever, not a redesign.

The rest of the notebook walks one prompt through all five.

## The prompt

> *"Design the serving backend for an AI coding assistant. Developers type in their IDE and get inline code completions. We have about 10,000 daily active users. Go."*

Deliberately underspecified: that's the test. **Step 1 is to interrogate it**, not to start drawing.

## Step 1 — Scope & requirements (ask, don't assume)

Questions you'd ask, and the assumptions you'd state and move on if the interviewer waves you off:

- **Interactive or batch?** Inline completions → *interactive*. **TTFT is the metric that matters** (the dev is watching the cursor). Assume a target: **p95 TTFT < 500 ms**.
- **How much traffic, really?** 10k DAU is not 10k QPS. Assume each active dev triggers ~20 completions/hour during a ~3-hour peak window → estimate peak **QPS** in step 2.
- **How long are the outputs?** Inline completions are short: assume **~50 output tokens**, prompt (surrounding code) **~1,000 tokens**. This dominates the KV-cache math.
- **Quality bar & model?** Assume a code-tuned model that fits one modern GPU (e.g. a 7B in fp16, or larger quantized). *Model choice is a quality-vs-cost lever, not the first decision.*
- **Cost ceiling?** Always ask. Assume "keep GPU spend reasonable" → we'll report a number and a knob to cut it.

> **🔵 Interview signal —** leading with *"interactive or batch, and what's the latency target?"* is the exact instinct 09/02 drilled. It tells the interviewer you know the throughput/latency trade governs every downstream choice.

## Step 2 — Back-of-envelope sizing (the numbers)

This is the 09/02 arithmetic, applied. Do it out loud, rounding freely.

**Peak QPS.** 10k DAU, ~⅓ active in the peak hour (~3.3k), ~20 completions/hour each ≈ 66k/hour ≈ **~18 QPS average**, so assume **~40 QPS peak** (bursty, devs cluster). Round to **40 QPS**.

**Throughput demand.** 40 QPS × 50 output tokens = **2,000 output tok/s**. (Prefill of the 1k-token prompt matters for TTFT, but decode tokens set steady-state throughput.)

**GPUs.** From 09/02: `GPUs = demand ÷ (throughput_per_gpu × utilization)`. Assume a code-model GPU does **~2,500 batched tok/s** and we run at **70% util** for headroom → 2,000 ÷ (2,500 × 0.7) ≈ **~1.15 → 2 GPUs** for compute. Round **up**, and you'd run ≥2 anyway for redundancy.

**VRAM sanity check.** Weights (7B fp16 ≈ 14 GB) + KV cache. From 09/02, a 1k-context request is light (~0.5 GB), so a 24–40 GB GPU batches plenty. **Not memory-bound here**, good: compute/QPS is the real constraint.

**Cost.** ~2 GPUs × ~$2/GPU-hr × 730 hr ≈ **~$3k/month** at this scale, before autoscaling down off-peak, which roughly halves it. *State it; it's the number the business cares about.*

> **⚠️ Production reality —** these numbers are *rough on purpose* and you say so. The point is the right **order of magnitude** and knowing **which input each output is sensitive to** (halve output length → halve GPU count; 10× users → 10× GPUs). Precision comes from load testing, not the whiteboard.

## Step 3 — Architecture (boxes and arrows)

Now the diagram, and every box traces to a section of this repo:

```
 IDE client ─▶ API gateway ─▶ request queue ─▶ ┌─ model server (vLLM) ─┐
 (dedup,          (auth,        (smooths        │  ...replica 2         │ ─▶ completion
  debounce)        rate-limit)   bursts)        └─ ...replica N (HPA) ──┘
                       │                              ▲        │
                   prompt cache ──────────────────────┘        ▼
                   (repeated prefixes)              observability + evals
                                                    (08/01, 02/04)
```

- **IDE client:** debounce keystrokes and cancel superseded requests. *The cheapest capacity is the request you never send.*
- **API gateway:** auth, rate-limit, route. The stable seam; behind it, you can switch the model layer without touching callers (09/01).
- **Request queue:** absorbs the bursts step 2 flagged. Interactive traffic is spiky; the queue is what keeps p95 TTFT honest under a spike instead of dropping requests.
- **Model servers (vLLM), N replicas:** the ~2 GPUs from step 2, behind an autoscaler (e.g. HPA on queue depth / GPU util). Continuous batching lives here (09/02).
- **Prompt cache:** coding prompts share huge prefixes (the same file, imports). Caching repeated prefixes cuts prefill work → better TTFT and throughput.
- **Observability + evals:** trace every call (08/01); run the offline eval set (02/04) on model/quantization changes *before* they ship.

## Step 4 — Trade-offs & bottlenecks (what breaks first)

Name the failure modes before the interviewer does. This is the senior signal.

- **First bottleneck: TTFT under burst.** When devs cluster, batching raises per-request latency (09/02's trade). Mitigation: cap batch size for this interactive service, keep utilization headroom, scale on queue depth, and accept a slightly higher GPU bill to protect TTFT.
- **Cost vs quality knob: the model.** A bigger/higher-quality model raises TTFT and GPU count. A quantized model (09/02) cuts both, at a quality cost you'd *measure* with the 02/04 evals. This is the main dial and you name it as one.
- **Cost vs latency: autoscaling.** Scaling to zero off-peak saves money but adds cold-start latency. For a dev tool with clear working hours, scale *down* (not to zero) overnight: a defensible middle.
- **Reliability (08/02):** a dead replica, a provider blip. Health checks + the queue + N≥2 replicas mean one failure degrades latency, not availability.
- **Security (07):** completions run on user code, so treat prompts as untrusted and don't log raw proprietary code (ties to 08/01's safe-logging).

## Step 5 — The follow-ups (each is a lever, not a redesign)

The interviewer will push. Because you sized it, each answer is one move on a number you already have:

| They say… | You reach for… (the lever) |
|---|---|
| "Now 100k users." | 10× demand → ~10× GPUs (step 2 scales linearly); revisit whether the queue/gateway tier scales too. |
| "Cut cost in half." | Quantize the model (09/02) → fewer/cheaper GPUs; + aggressive off-peak scale-down. Name the quality check (02/04). |
| "p99 TTFT spiked." | Batch size too high for interactive, or a hot replica. Cap batch, add headroom, check queue depth: the step-4 bottleneck. |
| "Longer context (whole-file)." | KV cache grows → batch shrinks → throughput drops → more GPUs (the 09/02 exercise). A capacity decision, not free. |
| "Multi-region / lower global latency." | Replicate the stack per region behind geo-routing; the gateway seam makes this additive, not a rewrite. |

> **🚩 Common mistake —** treating each follow-up as a new design. They're perturbations of the *same* sized system. Candidates who re-draw everything look like they never understood which number drives which box. You already did the math, so you just move the affected lever.

## The one-slide recap

The whole method, compressed to what you'd actually keep on the whiteboard:

```
1. REQUIREMENTS  interactive? latency target? QPS? cost ceiling? quality bar?
2. SIZING        QPS → tok/s → GPUs = demand ÷ (tok/s/gpu × util);  VRAM check;  $ / month
3. ARCHITECTURE  client → gateway → queue → vLLM replicas (autoscaled) → cache;  + obs/evals
4. BOTTLENECKS   TTFT under burst · model = cost/quality dial · autoscale = cost/latency
5. FOLLOW-UPS    each is one lever on a number you already have — not a redesign
```

> **⭐ Key takeaway —** you were never missing the pieces; you built them across sections 00–09. System design is the round where you *compose* them under pressure, lead with requirements and math, and defend the trade-offs. That composition, spoken clearly, is the AI-systems hire signal.

## Exercises

1. **Swap the prompt, keep the method.** Re-run all five steps for a *different* service: "design the backend for a customer-support RAG chatbot, 500 concurrent users." Note where the numbers and the architecture diverge from the coding-assistant (hint: longer contexts, retrieval hop, throughput over TTFT).
2. **Defend a number.** Someone challenges "why 2 GPUs, not 1?" Write the two-sentence answer that cites your step-2 math *and* the redundancy reason. This is the exact exchange the round tests.
3. **Find your own bottleneck.** For the capstone you're planning (section 12), do steps 1–2 only, then name the single resource that breaks first at 10× load and the lever you'd pull. One paragraph.
4. **The cost cut.** Take the worked example and produce a version at roughly *half* the monthly cost. List every lever you pulled and, for each, the risk you accepted, then say which one you'd reach for first and why.